In [1]:
import pandas as pd
from feast import FeatureStore

In [2]:
# Feature Store
store = FeatureStore("../03_feature_store/")

In [3]:
store

FeatureStore(
    repo_path=WindowsPath('../03_feature_store'),
    config=RepoConfig(project='ieee_cis_fraud', project_description=None, provider='local', registry_config='data/registry.db', online_config={'type': 'sqlite', 'path': 'data/online_store.db'}, auth={'type': 'no_auth'}, offline_config='dask', batch_engine_config='local', feature_server=None, flags=None, repo_path=WindowsPath('../03_feature_store'), entity_key_serialization_version=3, coerce_tz_aware=True, materialization_config=MaterializationConfig(pull_latest_features=False, online_write_batch_size=None), openlineage_config=None, mlflow_config=None, data_quality_monitoring_config=None),
    registry=not loaded,
    provider=not loaded
)

In [4]:
# Pegando as features
fv = store.get_feature_view(
    "transaction_features"
)

features = [
    f"transaction_features:{field.name}"
    for field in fv.schema
    if field.name not in {
        "transaction_id",
        "transaction_time",
        "label"
    }
]

In [5]:
# dataframe com os ids necessarios
df_entity = pd.read_parquet(
    '../00_dataset/02_final_data/df_train.pq',
    columns=['transaction_id', 'transaction_time', 'label']
)

In [6]:
df_entity.head()

,transaction_id,transaction_time,label
0,2994538,2017-12-03 18:50:58,0
1,2994542,2017-12-03 18:52:10,0
2,2994544,2017-12-03 18:52:29,0
3,2994552,2017-12-03 18:55:49,0
4,2994553,2017-12-03 18:55:53,0


In [7]:
# df training
df_training = store.get_historical_features(
    entity_df=df_entity,
    features=features,
).to_df()

Using transaction_time as the event timestamp. To specify a column explicitly, please name it event_timestamp.


In [9]:
df_training.head()

,transaction_id,transaction_time,label,numerical__d9,catgorical__product_cd,catgorical__device_info,numerical__d7,catgorical__m4,catgorical__id_31,catgorical__id_15,...,numerical__d12,numerical__addr2,numerical__id_10,catgorical__id_38,catgorical__r_emaildomain,numerical__addr1,numerical__id_09,numerical__d14,catgorical__id_16,catgorical__id_29
0,2987000,2017-12-02 00:00:00+00:00,0,-999.0,0.020523,0.025662,-999.0,0.113490,0.021198,0.021117,...,-999.0,87.0,-999.0,0.021117,0.020969,315.0,-999.0,-999.0,0.022907,0.021127
1,2987001,2017-12-02 00:00:01+00:00,0,-999.0,0.020350,0.025479,-999.0,0.037018,0.021004,0.020918,...,-999.0,87.0,-999.0,0.020918,0.020748,325.0,-999.0,-999.0,0.022725,0.020924
2,2987002,2017-12-02 00:01:09+00:00,0,-999.0,0.020350,0.025479,-999.0,0.037018,0.021004,0.020918,...,-999.0,87.0,-999.0,0.020918,0.020748,330.0,-999.0,-999.0,0.022725,0.020924
3,2987003,2017-12-02 00:01:39+00:00,0,-999.0,0.020523,0.025662,-999.0,0.036351,0.021198,0.021117,...,-999.0,87.0,-999.0,0.021117,0.020969,476.0,-999.0,-999.0,0.022907,0.021127
4,2987004,2017-12-02 00:01:46+00:00,0,-999.0,0.048630,0.000000,-999.0,0.018378,0.068579,0.048881,...,-999.0,87.0,-999.0,0.059340,0.020748,420.0,-999.0,-999.0,0.047920,0.051169


# Treinando um Benchmark

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TunedThresholdClassifierCV
from sklearn.metrics import classification_report

In [13]:
LABEL = 'label'
FEATURES = [
    field.name
    for field in fv.schema
    if field.name not in {
        "transaction_id",
        "transaction_time",
        "label"
    }
]

In [16]:
df_train, df_eval = train_test_split(
    df_training,
    shuffle=True,
    stratify=df_training[LABEL],
    test_size=0.2
)

In [17]:
df_train[LABEL].value_counts(normalize=True)

label
0    0.965011
1    0.034989
Name: proportion, dtype: float64

In [18]:
df_eval[LABEL].value_counts(normalize=True)

label
0    0.965007
1    0.034993
Name: proportion, dtype: float64

In [19]:
X_train, y_train = df_train[FEATURES], df_train[LABEL]
X_eval, y_eval = df_eval[FEATURES], df_eval[LABEL]

In [22]:
# model
model = RandomForestClassifier(n_estimators=100, max_depth=7, min_samples_split=20)

In [23]:
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,7
,min_samples_split,20
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [33]:
model_tuned = TunedThresholdClassifierCV(
    RandomForestClassifier(n_estimators=100, max_depth=7, min_samples_split=20),
    scoring="f1"
)

In [34]:
model_tuned.fit(X_train, y_train)

,estimator,RandomForestC...ples_split=20)
,scoring,'f1'
,response_method,'auto'
,thresholds,100
,cv,None
,refit,True
,n_jobs,None
,random_state,None
,store_cv_results,False
,n_estimators,100
,criterion,'gini'


In [35]:
model_tuned

,estimator,RandomForestC...ples_split=20)
,scoring,'f1'
,response_method,'auto'
,thresholds,100
,cv,None
,refit,True
,n_jobs,None
,random_state,None
,store_cv_results,False
,n_estimators,100
,criterion,'gini'


In [31]:
print(classification_report(y_eval, model.predict(df_eval[FEATURES])))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98    113975
           1       0.85      0.06      0.12      4133

    accuracy                           0.97    118108
   macro avg       0.91      0.53      0.55    118108
weighted avg       0.96      0.97      0.95    118108



In [36]:
print(classification_report(y_eval, model_tuned.predict(df_eval[FEATURES])))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98    113975
           1       0.37      0.27      0.31      4133

    accuracy                           0.96    118108
   macro avg       0.67      0.62      0.64    118108
weighted avg       0.95      0.96      0.96    118108



# Salvando o modelo

In [39]:
import joblib
from pathlib import Path

In [40]:
ARTFACTS = Path('../artfacts')

In [42]:
joblib.dump(model, ARTFACTS / 'model_mvp_v0.joblib')
joblib.dump(model_tuned, ARTFACTS / 'model_mvp_threshould_tuned_v0.joblib')

['..\\artfacts\\model_mvp_threshould_tuned_v0.joblib']